# exp069c: Hybrid pseudo merge (3 NPZ → 3 pseudo)

**Inputs (kernel_sources)**:
  - exp069a: Babych ensemble (206 BC25 class)
  - exp069b-nb4, tucker, exp029: 3 stream raw (234 BC26 class)

**Process**:
  1. exp069b 3 stream → exp048 blend (rank-avg + sonotype mirror、234 class)
  2. exp069a Babych ensemble → BC25→BC26 species mapping (inat_taxon_id 照合)
  3. Per-class merge:
     - pseudo_babych_234.npz: Babych mapped (non-overlap NaN)
     - pseudo_exp048_234.npz: exp048 blend
     - **pseudo_hybrid_234.npz**: ★ overlap=Babych, non-overlap=exp048 ★

**Output → /kaggle/working/**:
  - pseudo_babych_234.npz, pseudo_exp048_234.npz, pseudo_hybrid_234.npz
  - bc25_to_bc26_map.json: species mapping reference

**M-R3 usage**:
  - M1, M2 (Perch ON): pseudo_exp048_234.npz
  - M3-M7 (Babych init): pseudo_hybrid_234.npz


In [ ]:
# Setup
import sys, os, time, json
from pathlib import Path
import numpy as np
import pandas as pd
print(f"Python: {sys.version[:50]}")
print(f"numpy: {np.__version__}")
START = time.time()


In [ ]:
# Locate inputs
def find_dir(candidates):
    for p in candidates:
        if Path(p).exists():
            return Path(p)
    return None

# BC2026 competition data
DATA_PATH = find_dir([
    "/kaggle/input/competitions/birdclef-2026",
    "/kaggle/input/birdclef-2026",
])
assert DATA_PATH is not None
TAXONOMY_CSV = DATA_PATH / "taxonomy.csv"

# exp069a output (Babych ensemble)
EXP069A_DIR = find_dir([
    "/kaggle/input/birdclef2026-exp069a-babych-pseudo-gen",   # if added as kernel_source
    "/kaggle/input/notebooks/maekeso/birdclef2026-exp069a-babych-pseudo-gen",
])
assert EXP069A_DIR is not None, "exp069a kernel output not attached"

# exp069b 3 stream outputs
EXP069B_NB4_DIR = find_dir([
    "/kaggle/input/birdclef2026-exp069b-nb4-pseudo",
    "/kaggle/input/notebooks/maekeso/birdclef2026-exp069b-nb4-pseudo",
])
EXP069B_TUCKER_DIR = find_dir([
    "/kaggle/input/birdclef2026-exp069b-tucker-pseudo",
    "/kaggle/input/notebooks/maekeso/birdclef2026-exp069b-tucker-pseudo",
])
EXP069B_EXP029_DIR = find_dir([
    "/kaggle/input/birdclef2026-exp069b-exp029-pseudo",
    "/kaggle/input/notebooks/maekeso/birdclef2026-exp069b-exp029-pseudo",
])
assert EXP069B_NB4_DIR is not None, "exp069b-nb4 not attached"
assert EXP069B_TUCKER_DIR is not None, "exp069b-tucker not attached"
assert EXP069B_EXP029_DIR is not None, "exp069b-exp029 not attached"

OUT_DIR = Path("/kaggle/working")

print(f"DATA_PATH: {DATA_PATH}")
print(f"EXP069A_DIR: {EXP069A_DIR}")
print(f"EXP069B_NB4: {EXP069B_NB4_DIR}")
print(f"EXP069B_TUCKER: {EXP069B_TUCKER_DIR}")
print(f"EXP069B_EXP029: {EXP069B_EXP029_DIR}")


In [ ]:
# Load BC26 taxonomy
taxo = pd.read_csv(TAXONOMY_CSV)
PRIMARY_LABELS = taxo["primary_label"].astype(str).tolist()
N_CLASSES_BC26 = len(PRIMARY_LABELS)
assert N_CLASSES_BC26 == 234, f"Expected 234 BC26 classes, got {N_CLASSES_BC26}"
print(f"BC26 species: {N_CLASSES_BC26}")

# inat_taxon_id mapping
bc26_taxon_id_to_idx = {}
for idx, row in taxo.iterrows():
    tid = str(row["inat_taxon_id"])
    bc26_taxon_id_to_idx[tid] = idx
print(f"BC26 inat_taxon_id mapping: {len(bc26_taxon_id_to_idx)}")

# Load exp069a Babych label2ind (BC25 species → 206 index)
babych_label2ind_path = EXP069A_DIR / "babych_label2ind.json"
assert babych_label2ind_path.exists()
babych_label2ind = json.loads(babych_label2ind_path.read_text())
print(f"Babych BC25 species: {len(babych_label2ind)}")

# BC25 ↔ BC26 mapping (by inat_taxon_id where applicable)
# Babych label format: many are inat_taxon_id strings, some are codes
# Build overlap pairs
bc25_to_bc26_pairs = []
for bc25_label, bc25_idx in babych_label2ind.items():
    if bc25_label in bc26_taxon_id_to_idx:
        bc26_idx = bc26_taxon_id_to_idx[bc25_label]
        bc25_to_bc26_pairs.append((bc25_idx, bc26_idx, bc25_label))
print(f"BC25 ↔ BC26 overlap: {len(bc25_to_bc26_pairs)} species")


In [ ]:
# Load 3 exp069b stream NPZ
def load_npz(dir_path, fname):
    p = dir_path / fname
    if not p.exists():
        # Try alternative
        candidates = list(dir_path.glob("*.npz"))
        if candidates:
            p = candidates[0]
        else:
            raise FileNotFoundError(f"NPZ not found in {dir_path}")
    print(f"  Loading {p}")
    return dict(np.load(p, allow_pickle=True))

print("=== Loading 3 stream NPZ ===")
nb4_data = load_npz(EXP069B_NB4_DIR, "nb4_raw_234.npz")
tucker_data = load_npz(EXP069B_TUCKER_DIR, "tucker_raw_234.npz")
exp029_data = load_npz(EXP069B_EXP029_DIR, "exp029_raw_234.npz")

probs_nb4 = nb4_data["probs"].astype(np.float32)
probs_tucker = tucker_data["probs"].astype(np.float32)
probs_exp029 = exp029_data["probs"].astype(np.float32)

file_ids_nb4 = nb4_data["file_ids"]
file_ids_tucker = tucker_data["file_ids"]
file_ids_exp029 = exp029_data["file_ids"]

print(f"NB4 v11:    {probs_nb4.shape}, files={len(file_ids_nb4)}")
print(f"Tucker SED: {probs_tucker.shape}, files={len(file_ids_tucker)}")
print(f"exp029 R3:  {probs_exp029.shape}, files={len(file_ids_exp029)}")

# Verify alignment
assert np.array_equal(file_ids_nb4, file_ids_tucker), "file_ids mismatch nb4/tucker"
assert np.array_equal(file_ids_nb4, file_ids_exp029), "file_ids mismatch nb4/exp029"
file_ids = file_ids_nb4
N_FILES, N_WINDOWS, N_CLASSES = probs_nb4.shape


In [ ]:
# exp048 blend logic (3-way rank avg + sonotype mirror)
# Weights from exp048: 0.30 / 0.40 / 0.30
W_NB4 = 0.30
W_TUCKER = 0.40
W_EXP029 = 0.30

print("=== Building exp048 blend ===")

# Flatten for rank computation
flat_nb4 = probs_nb4.reshape(-1, N_CLASSES)
flat_tucker = probs_tucker.reshape(-1, N_CLASSES)
flat_exp029 = probs_exp029.reshape(-1, N_CLASSES)

# Rank-avg blend (per-class, percentile rank)
rank_nb4 = pd.DataFrame(flat_nb4).rank(axis=0, pct=True).to_numpy().astype(np.float32)
rank_tucker = pd.DataFrame(flat_tucker).rank(axis=0, pct=True).to_numpy().astype(np.float32)
rank_exp029 = pd.DataFrame(flat_exp029).rank(axis=0, pct=True).to_numpy().astype(np.float32)
blend_flat = W_NB4 * rank_nb4 + W_TUCKER * rank_tucker + W_EXP029 * rank_exp029

# Sonotype mirror (exp048 spec)
MIRROR_PAIRS = (
    ("47158son15", "47158son16"),
    ("47158son09", "47158son12"),
    ("47158son02", "47158son14"),
    ("47158son13", "47158son21", "47158son22", "47158son23"),
)
col_to_idx = {lbl: i for i, lbl in enumerate(PRIMARY_LABELS)}
mirror_count = 0
for group in MIRROR_PAIRS:
    valid_idx = [col_to_idx[s] for s in group if s in col_to_idx]
    if len(valid_idx) >= 2:
        group_max = blend_flat[:, valid_idx].max(axis=1, keepdims=True)
        blend_flat[:, valid_idx] = group_max
        mirror_count += len(valid_idx)
print(f"Sonotype mirror applied to {mirror_count} columns")

# Reshape back
probs_exp048 = blend_flat.reshape(N_FILES, N_WINDOWS, N_CLASSES).astype(np.float16)
print(f"exp048 blend: {probs_exp048.shape}, mean={probs_exp048.mean():.4f}, max={probs_exp048.max():.4f}")


In [ ]:
# Map Babych ensemble (206 BC25) → 234 BC26 (overlap species のみ、non-overlap NaN)
# ★ V2 fix: Babych raw prob (sigmoid) を rank-pct に変換して exp048 と scale 統一
babych_npz = dict(np.load(EXP069A_DIR / "babych_ensemble_206.npz", allow_pickle=True))
babych_probs = babych_npz["probs"].astype(np.float32)   # (n_files, n_windows, 206)
babych_file_ids = babych_npz["file_ids"]
print(f"Babych ensemble (raw prob): {babych_probs.shape}, files={len(babych_file_ids)}")
print(f"  Raw prob stats: mean={babych_probs.mean():.4f}, std={babych_probs.std():.4f}, max={babych_probs.max():.4f}")

# Verify file_ids alignment (must match exp069b streams)
assert np.array_equal(babych_file_ids, file_ids), "Babych vs exp069b file_ids mismatch"

# ★ V2 fix: rank-pct 変換 (per-class column-wise、exp048 と同 scale [0,1] uniform に統一)
# Babych raw prob はsigmoid 出力で極端に skew (mean ~0.05)、
# exp048 は rank-pct uniform (mean ~0.5)
# scale gap 10x → 同 model に渡すと calibration 崩壊するため、Babych も rank-pct に変換
n_babych_classes = babych_probs.shape[2]
babych_flat = babych_probs.reshape(-1, n_babych_classes)
babych_rank_flat = pd.DataFrame(babych_flat).rank(axis=0, pct=True).to_numpy().astype(np.float32)
babych_rank = babych_rank_flat.reshape(N_FILES, N_WINDOWS, n_babych_classes)
print(f"  Rank-pct stats: mean={babych_rank.mean():.4f}, std={babych_rank.std():.4f}, max={babych_rank.max():.4f}")
print(f"  ★ Scale unified with exp048 (both rank-pct [0,1] uniform)")

# Create mapped tensor (NaN for non-overlap)
probs_babych_mapped = np.full((N_FILES, N_WINDOWS, N_CLASSES), np.nan, dtype=np.float32)
for bc25_idx, bc26_idx, _ in bc25_to_bc26_pairs:
    probs_babych_mapped[:, :, bc26_idx] = babych_rank[:, :, bc25_idx]   # ★ rank 使用

# Verify coverage
overlap_classes = sorted([bc26_idx for _, bc26_idx, _ in bc25_to_bc26_pairs])
non_overlap_classes = [i for i in range(N_CLASSES) if i not in set(overlap_classes)]
print(f"Babych mapped (rank-pct): {len(overlap_classes)} overlap (no NaN), {len(non_overlap_classes)} non-overlap (NaN)")


In [ ]:
# Hybrid: overlap = Babych (rank-pct)、non-overlap = exp048 (rank-pct)
probs_hybrid = probs_exp048.astype(np.float32).copy()
for bc26_idx in overlap_classes:
    probs_hybrid[:, :, bc26_idx] = probs_babych_mapped[:, :, bc26_idx]

# Quality check: NaN should not exist in hybrid
n_nan = np.isnan(probs_hybrid).sum()
assert n_nan == 0, f"Hybrid has {n_nan} NaN values - bug!"

print(f"Hybrid pseudo: {probs_hybrid.shape}")
print(f"  Overlap ({len(overlap_classes)} cls, Babych rank): mean={probs_hybrid[:,:,overlap_classes].mean():.4f}")
print(f"  Non-overlap ({len(non_overlap_classes)} cls, exp048 rank): mean={probs_hybrid[:,:,non_overlap_classes].mean():.4f}")

# ============================================================
# ★ V2 Verification: scale 統一 + distribution sanity check
# ============================================================
print("\n" + "="*60)
print("=== V2 Verification: Scale & Distribution Sanity ===")
print("="*60)

# 1. Scale comparison (V1 vs V2 想定)
overlap_part = probs_hybrid[:, :, overlap_classes].flatten()
nonoverlap_part = probs_hybrid[:, :, non_overlap_classes].flatten()

print("\n[Scale comparison]")
print(f"  Overlap (Babych rank):    mean={overlap_part.mean():.4f} std={overlap_part.std():.4f} median={np.median(overlap_part):.4f}")
print(f"  Non-overlap (exp048):     mean={nonoverlap_part.mean():.4f} std={nonoverlap_part.std():.4f} median={np.median(nonoverlap_part):.4f}")
scale_gap = abs(overlap_part.mean() - nonoverlap_part.mean())
print(f"  Mean gap: {scale_gap:.4f}  (V1=~0.45 mismatch, V2 target<0.05)")
if scale_gap < 0.05:
    print(f"  ✓ Scale unified (V2 fix successful)")
else:
    print(f"  ★ WARNING: Scale gap still large, V2 fix may have failed")

# 2. Percentile distribution (rank-pct なら uniform 期待 → percentile ratio ~10% each)
print("\n[Distribution percentiles - should be ~0.1, 0.3, 0.5, 0.7, 0.9 for uniform]")
for name, part in [("Overlap (Babych)", overlap_part), ("Non-overlap (exp048)", nonoverlap_part)]:
    p10, p30, p50, p70, p90 = np.percentile(part, [10, 30, 50, 70, 90])
    print(f"  {name:30s}: p10={p10:.3f} p30={p30:.3f} p50={p50:.3f} p70={p70:.3f} p90={p90:.3f}")

# 3. Per-class spot check (Babych でラベル知識ある species を 5 つ pick)
print("\n[Per-class spot check (overlap species)]")
sample_overlap = overlap_classes[:5] if len(overlap_classes) >= 5 else overlap_classes
for bc26_idx in sample_overlap:
    species = PRIMARY_LABELS[bc26_idx]
    col = probs_hybrid[:, :, bc26_idx].flatten()
    p95 = np.percentile(col, 95)
    p99 = np.percentile(col, 99)
    print(f"  bc26_idx={bc26_idx:3d} ({species}): mean={col.mean():.3f} p95={p95:.3f} p99={p99:.3f} max={col.max():.3f}")

# 4. Range check
print(f"\n[Range check]")
print(f"  Hybrid global: min={probs_hybrid.min():.4f} max={probs_hybrid.max():.4f}")
print(f"  Expected: [0, 1] (rank-pct)")
assert probs_hybrid.min() >= 0.0 and probs_hybrid.max() <= 1.0, "Hybrid values out of [0,1]!"
print(f"  ✓ All values in [0, 1]")

# 5. After-power-transform preview (k=1.54 が train 時に適用される)
PSEUDO_POWER_K_PREVIEW = 1.54
hybrid_k = probs_hybrid ** PSEUDO_POWER_K_PREVIEW
overlap_k_mean = hybrid_k[:, :, overlap_classes].mean()
nonoverlap_k_mean = hybrid_k[:, :, non_overlap_classes].mean()
print(f"\n[Power transform k=1.54 preview (M3-M6 が train 時に適用)]")
print(f"  Overlap after k:    mean={overlap_k_mean:.4f}")
print(f"  Non-overlap after k: mean={nonoverlap_k_mean:.4f}")
print(f"  Gap after k: {abs(overlap_k_mean - nonoverlap_k_mean):.4f}  (target<0.05)")

probs_hybrid_f16 = probs_hybrid.astype(np.float16)


In [ ]:
# Save all outputs to /kaggle/working/
import json as _json

print("=== Saving NPZ outputs ===")

# 1. Babych mapped (with NaN for non-overlap)
np.savez_compressed(
    OUT_DIR / "pseudo_babych_234.npz",
    probs=probs_babych_mapped.astype(np.float16),
    file_ids=file_ids,
)
print(f"  pseudo_babych_234.npz: {(OUT_DIR / 'pseudo_babych_234.npz').stat().st_size/1e6:.1f} MB")

# 2. exp048 blend (full 234)
np.savez_compressed(
    OUT_DIR / "pseudo_exp048_234.npz",
    probs=probs_exp048,
    file_ids=file_ids,
)
print(f"  pseudo_exp048_234.npz: {(OUT_DIR / 'pseudo_exp048_234.npz').stat().st_size/1e6:.1f} MB")

# 3. Hybrid (overlap=Babych, non-overlap=exp048)
np.savez_compressed(
    OUT_DIR / "pseudo_hybrid_234.npz",
    probs=probs_hybrid_f16,
    file_ids=file_ids,
)
print(f"  pseudo_hybrid_234.npz: {(OUT_DIR / 'pseudo_hybrid_234.npz').stat().st_size/1e6:.1f} MB")

# Metadata
with open(OUT_DIR / "primary_labels.json", "w") as f:
    _json.dump(list(PRIMARY_LABELS), f, indent=2)

bc25_bc26_map = {
    "n_overlap": len(bc25_to_bc26_pairs),
    "overlap_pairs": [{"bc25_idx": p[0], "bc26_idx": p[1], "label": p[2]} for p in bc25_to_bc26_pairs],
    "overlap_bc26_indices": overlap_classes,
    "non_overlap_bc26_indices": non_overlap_classes,
    "blend_weights": {"nb4": W_NB4, "tucker": W_TUCKER, "exp029": W_EXP029},
}
with open(OUT_DIR / "bc25_to_bc26_map.json", "w") as f:
    _json.dump(bc25_bc26_map, f, indent=2)

with open(OUT_DIR / "file_index.json", "w") as f:
    _json.dump({str(fid): i for i, fid in enumerate(file_ids)}, f)

with open(OUT_DIR / "exp069c_summary.json", "w") as f:
    _json.dump({
        "n_files": int(N_FILES),
        "n_windows": int(N_WINDOWS),
        "n_classes_bc26": int(N_CLASSES),
        "n_overlap_species": len(overlap_classes),
        "n_non_overlap_species": len(non_overlap_classes),
        "total_time_min": (time.time() - START) / 60,
    }, f, indent=2)

print(f"\nOK exp069c DONE: total {(time.time()-START)/60:.1f} min")
